# ArcGIS Server Map Project Inventory

This notebook accesses the ArcGIS Server Administrator to search for services, retrieves their source project paths (.mxd or .aprx), and exports the data to a CSV file. This is helpful for tracking which projects are associated with the published service.

## Requirements
- ArcGIS Server Admin API access
- Python libraries: requests, json, csv, getpass, urllib3

## Usage
1. Update the `server_base_url` in the main function. Example 'http://[your server IP]:6080/arcgis'
2. Run the cells in order.
3. Enter your ArcGIS Server admin credentials when prompted.

In [ ]:
import requests
import json
import csv
import getpass
import urllib3

# Suppress insecure request warnings if using self-signed certificates
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
def get_token(admin_url, username, password):
    """Generate a token for ArcGIS Server Admin API."""
    token_url = f"{admin_url}/generateToken"
    params = {
        'username': username,
        'password': password,
        'client': 'requestip',
        'f': 'json'
    }
    try:
        response = requests.post(token_url, data=params, verify=False)
        response.raise_for_status()
        token_data = response.json()
        if 'token' in token_data:
            return token_data['token']
        else:
            print("Failed to generate token. Check credentials.")
            return None
    except Exception as e:
        print(f"Error generating token: {e}")
        return None

In [ ]:
def get_service_manifest(admin_url, token, folder, service_name, service_type):
    """Retrieve the manifest.json for a specific service to find the source document."""
    if folder:
        manifest_url = f"{admin_url}/services/{folder}/{service_name}.{service_type}/iteminfo/manifest/manifest.json"
    else:
        manifest_url = f"{admin_url}/services/{service_name}.{service_type}/iteminfo/manifest/manifest.json"
    
    params = {'token': token, 'f': 'json'}
    try:
        response = requests.get(manifest_url, params=params, verify=False)
        return response.json()
    except Exception as e:
        print(f"Error fetching manifest for {service_name}: {e}")
        return {}

In [ ]:
def extract_project_path(manifest_data):
    """Parse the manifest JSON to find the .mxd or .aprx path."""
    if 'resources' in manifest_data:
        for resource in manifest_data['resources']:
            if 'onPremisePath' in resource:
                path = resource['onPremisePath']
                # Check if the resource is an ArcMap or ArcGIS Pro project file
                if path.lower().endswith(('.mxd', '.aprx')):
                    return path
    return "Path not found in manifest"

In [ ]:
def main():
    # --- Configuration ---
    # CHANGE THIS to your exact base URL
    # Example SERVER ADMIN BASE URL: "http://<your-server>:6080/arcgis"
    server_base_url = "[YOUR SERVER ADMIN BASE URL GOES HERE]" # Change to your server URL
    admin_url = f"{server_base_url}/admin"
    output_csv = "ArcGISServer_Projects.csv"
    
    print("--- ArcGIS Server Map Service Inventory ---")
    username = input("Enter ArcGIS Server Admin Username: ")
    password = getpass.getpass("Enter ArcGIS Server Admin Password: ")

    token = get_token(admin_url, username, password)
    if not token:
        return

    params = {'token': token, 'f': 'json'}
    services_inventory = []

    print("\nFetching folders and services...")
    
    # 1. Get Root level services and folders
    root_response = requests.get(f"{admin_url}/services", params=params, verify=False).json()
    folders = root_response.get('folders', [])
    root_services = root_response.get('services', [])

    # Process Root Services (only MapServer for this script)
    for service in root_services:
        if service['type'] == 'MapServer':
            print(f"Processing (Root): {service['serviceName']}")
            manifest = get_service_manifest(admin_url, token, None, service['serviceName'], service['type'])
            project_path = extract_project_path(manifest)
            services_inventory.append({
                'Folder': 'Root',
                'Service Name': service['serviceName'],
                'Service URL': f"{server_base_url}/rest/services/{service['serviceName']}/MapServer",
                'Project Path (.mxd/.aprx)': project_path
            })

    # 2. Iterate through Folders
    for folder in folders:
        # Utilities and System folders usually don't have published user projects, but you can remove this check if needed
        if folder in ['Utilities', 'System']:
            continue
            
        folder_response = requests.get(f"{admin_url}/services/{folder}", params=params, verify=False).json()
        folder_services = folder_response.get('services', [])
        
        for service in folder_services:
            if service['type'] == 'MapServer':
                print(f"Processing ({folder}): {service['serviceName']}")
                manifest = get_service_manifest(admin_url, token, folder, service['serviceName'], service['type'])
                project_path = extract_project_path(manifest)
                services_inventory.append({
                    'Folder': folder,
                    'Service Name': service['serviceName'],
                    'Service URL': f"{server_base_url}/rest/services/{folder}/{service['serviceName']}/MapServer",
                    'Project Path (.mxd/.aprx)': project_path
                })

    # 3. Write results to CSV
    if services_inventory:
        print(f"\nWriting results to {output_csv}...")
        with open(output_csv, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['Folder', 'Service Name', 'Service URL', 'Project Path (.mxd/.aprx)'])
            writer.writeheader()
            writer.writerows(services_inventory)
        print("Done!")
    else:
        print("\nNo Map Services found.")

In [ ]:
if __name__ == "__main__":
    main()